In [ ]:
import json
import statistics
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from huggingface_hub import hf_hub_download
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet50


In [ ]:
REPO_ID = "ibrahimdaud/raw-food-recognition-models"
CLASS_NAMES_PATH = Path("../class_names.json")
FINETUNED_PATH = Path("../model_training/finetuned_resnet500.pt")
DATA_FOLDER = Path("../data")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

The model comes from https://huggingface.co/ibrahimdaud/raw-food-recognition-models There is no model linked in but there is the parameters stored in binary. I took from torchvision the used model before putting the parameters. In hugging face, there is others, but the model can't be taken from scratch like I did, so I kept the basic model.

In [ ]:
checkpoint_path = hf_hub_download(
    repo_id=REPO_ID, filename="resnet50_pytorch_model.bin"
)
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

model = resnet50(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, 90)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval().to(device)

In [ ]:
state_dict = torch.load(FINETUNED_PATH, map_location="cpu", weights_only=True)

model_tuned = resnet50(weights=None)
model_tuned.fc = torch.nn.Linear(model_tuned.fc.in_features, 90)
model_tuned.load_state_dict(state_dict)
model_tuned.eval().to(device)

Class_names are generated with the file build_class_names.py because there is nothing in the metadata, so I generated it from the dataset. There is some very specific class name ( like a specific type of yogourt) that must be filtered out. There is also nothing about meat, cheese, mushrooms or fish, so there is missing a few things

In [ ]:
with open(CLASS_NAMES_PATH, encoding="utf-8") as f:
    class_names = json.load(f)
    print(len(class_names))

In [ ]:

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

The image is simply preprocessed, we should see if other processing is required

In [ ]:
def recognize_food(image_path, top_k=5):
    start_time = time.time()
    image = Image.open(image_path).convert("RGB")
    tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.inference_mode():
        probs = torch.softmax(model(tensor), dim=1)[0]
        probs_tuned = torch.softmax(model_tuned(tensor), dim=1)[0]
    scores, indices = torch.topk(probs, min(top_k, 90))
    scores_tuned, indices_tuned = torch.topk(probs_tuned, min(top_k, 90))
    end_time = time.time()
    results =  [
        {"label": class_names[i.item()], "confidence": round(s.item(), 4)}
        for s, i in zip(scores, indices)
    ]
    tuned_result =  [{"label_tuned": class_names[i.item()], "confidence_tuned": round(s.item(), 4)}
            for s, i in zip(scores_tuned, indices_tuned)
            ]
    elapsed_time = (end_time - start_time)
    return results, tuned_result, elapsed_time

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

images = [
    path
    for path in DATA_FOLDER.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
]

In [20]:

times = []
for image_path in images:
    print(f"\nImage: {image_path}")
    print("-" * 50)
    results, results_tuned, time_taken = recognize_food(image_path)

    img = Image.open(image_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Image: {image_path}")
    for result in results:
        print(f"{result['label']:<25} {result['confidence']:.2%}")
    for tuned_result in results_tuned:
        print(f"  {tuned_result['label_tuned']:<25} {tuned_result['confidence_tuned']:.2%}")
    print(f"in {time_taken:.2f}s")
    plt.show()
    times.append(time_taken)
print(f"mean : {statistics.mean(times):.2f}s")

The result is taking in average 1.7 seconds to compute.